# SeCap reproduction: private datasets, GPU T4 ×2

Author source: `ryx9/SeCap-AGPReID-reproduce@df2d9dc75df35662e0e9afc444e14dec7ae4e6fc`. Source is embedded for reproducibility; no GitHub push is required. Enable Internet and **GPU T4 ×2**, attach your private dataset, then **Run All**. Setup uses an isolated Python 3.10 / PyTorch 2.2.2 CUDA 12.1 environment managed by uv; no manual Conda steps.

This run is intentionally scoped to **CARGO only**. The uploaded G2A-VReID is a different benchmark from the paper's G2APS-ReID and is not run here; LAGPeR is likewise excluded so the remaining GPU budget is devoted to CARGO.

Defaults follow the paper's batch 64 / stride 16 / LR 0.008 / 120 epochs. The released YAML differs (batch 128; CARGO and G2APS-ReID stride 12). The author's multi-GPU code mines triplets and computes BatchNorm within each GPU batch; with two GPUs, these operations see 32 images each, so exact numerical equivalence to the paper's single RTX 3090 run is not promised. Set batch 128 and the desired stride below to follow the released YAML instead. All overrides are saved.

Every epoch saves metrics and a full resumable checkpoint. Best means the author's first test protocol metric `(Rank-1 + mAP)/2`; evaluation follows the released interval of five epochs. The latest two epoch files, best, last, optimizer, schedulers, AMP scaler and per-rank RNG states are preserved. Old epoch checkpoints are pruned to fit Kaggle storage; **every epoch's results remain**. Resuming restores training state at an epoch boundary; worker/sampler prefetch means it is not bitwise identical to an uninterrupted run.

Private artifact dataset snapshots are uploaded hourly and at exit. Lightweight log/progress heartbeats are uploaded every 15 minutes for external monitoring. A hardlink-aware `checkpoints.tar` avoids storing duplicate copies of the best/last/epoch aliases; it is extracted automatically on resume. The notebook stops at an epoch boundary within a quota-aware budget (at most eight hours, with one GPU hour reserved for archiving), leaving time before Kaggle's session limit to archive. Run All again to restore the latest private archive and continue. Kaggle GPU quotas can require waiting before the next session. A previously attached archive dataset is also supported.


In [ ]:
SETTINGS = {
    "datasets": ["CARGO"],
    "artifact_dataset": "rizwanabdullah05/secap-reproduction-artifacts",
    "monitor_dataset": "rizwanabdullah05/secap-training-heartbeat",
    "batch_size": 64,
    "stride": [16, 16],
    "max_epoch": 90,
    "eval_period": 5,
    "session_hours": 8,
    "archive_minutes": 60,
}


## Restore the exact source and run setup/training
All image reads, environment installation, training, intermediate checkpoints and archive uploads happen on Kaggle. No API token is included in this notebook. Kagglehub uses the notebook account authentication.

In [ ]:
import base64, hashlib, io, tarfile
from pathlib import Path
SOURCE_B64 = '...'
source_bytes = base64.b64decode(SOURCE_B64)
assert hashlib.sha256(source_bytes).hexdigest() == '3a8fc10975650c0897dd892e7b6b656eaa6c54d5a1b3c42589113fdf738bf9bd'
Path('/tmp/secap-source.tar.gz').write_bytes(source_bytes)
repo = Path('/tmp/secap-source')
repo.mkdir(exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(source_bytes)) as bundle:
    bundle.extractall(repo, filter='data')
namespace = {'__name__': 'secap_kaggle_runtime'}
exec(compile((repo/'kaggle/runtime.py').read_text(), 'kaggle/runtime.py', 'exec'), namespace)
SETTINGS['source_commit'] = 'df2d9dc75df35662e0e9afc444e14dec7ae4e6fc'
namespace['main'](SETTINGS)


## Outputs and continuation
`/kaggle/working/secap-artifacts/<dataset>/` contains `epochs.jsonl`, the author's `metrics.json`, `config.yaml`, `progress.json`, `console.log`, plots, `model_best.pth`, `model_last.pth`, and the most recent epoch files. `model_final.pth` is written only at epoch 120. Exact source, environment freeze, pretrained weight hash and validation reports are included.

The private archive is `rizwanabdullah05/secap-reproduction-artifacts`. Re-run this notebook to continue from `model_last.pth`. Existing completed datasets are skipped. Keep the same batch and stride when resuming; use a different artifact dataset slug for an independent experiment. Final test results are recorded in the evaluation logs and epoch metrics. Best and final checkpoint evaluations can be run through the original `tools/train_net.py --eval-only` entry point using the saved configuration.
